In [1]:
import requests
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0 (DataScienceClass/1.0)"}
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Game_of_the_Year_awards_(board_games)"

r = requests.get(WIKI_URL, headers=HEADERS)
r.raise_for_status()
all_tables = pd.read_html(r.text)

AWARD_TABLE_MAP = {
    0:  ("American Tabletop Awards",            "Early Gamers"),
    1:  ("American Tabletop Awards",            "Casual Games"),
    2:  ("American Tabletop Awards",            "Strategy Games"),
    3:  ("American Tabletop Awards",            "Complex Games"),
    4:  ("As d'Or / Golden Ace",                "General"),
    5:  ("Board Game Quest Awards",             "General"),
    6:  ("Deutscher Spiele Preis",              "General"),
    7:  ("Diamond Climber Awards",              "General"),
    8:  ("Jogo do Ano",                         "General"),
    9:  ("Games magazine",                      "General"),
    10: ("Golden Geek Award (2020 onwards)",    "General"),
    11: ("Golden Geek Award (Till 2019)",       "General"),
    12: ("Le Diamant d'Or",                     "General"),
    13: ("Mensa Select",                        "General"),
    14: ("Premio JdA",                          "General"),
    15: ("Spiel des Jahres",                    "General"),
    16: ("Spiel des Jahre",                     "Kennerspiel des Jahres"),
    17: ("Spiel des Jahre",                     "Kinderspiel des Jahres"),
}

def clean_citations(series):
    """Remove Wikipedia citation brackets like [15], [n], [citation needed]."""
    return (
        series.astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )

WINNER_KEYWORDS = ["winner", "game", "title", "name"]

award_frames = []

for idx, (award_name, category) in AWARD_TABLE_MAP.items():
    if idx >= len(all_tables):
        print(f"  ⚠ Table {idx} not found — skipping {award_name} / {category}")
        continue

    t = all_tables[idx].copy()

    # Normalize column names to lowercase
    t.columns = [str(c).lower().strip() for c in t.columns]

    # Find year column
    year_col = next((c for c in t.columns if "year" in c), None)
    if not year_col:
        print(f"  ⚠ Table {idx} ({award_name}): no year column found — skipping")
        continue

    # Find winner/name column
    name_col = next(
        (c for c in t.columns if any(kw in c for kw in WINNER_KEYWORDS)), None
    )

    # ── ROUTE SELECTION ─────────────────────────────────────
    # Rule 1: tuple[1] != "General" → use hardcoded category
    # Rule 2: tuple[1] == "General" AND df has 'category' col → use that column
    # Rule 3: tuple[1] == "General" AND no winner col found   → melt()
    # Fallback: none of the above → skip with warning

    if category != "General":
        # ── RULE 1 ──────────────────────────────────────────
        if not name_col:
            print(f"  ⚠ [FALLBACK] Table {idx} ({award_name} / {category}): "
                  f"no winner column — columns={t.columns.tolist()}")
            continue
        result = t[[year_col, name_col]].copy()
        result.columns = ["year", "game_name"]
        result["category"] = category
        path = "Rule 1 (hardcoded category)"

    elif "category" in t.columns and name_col:
        # ── RULE 2 ──────────────────────────────────────────
        result = t[[year_col, "category", name_col]].copy()
        result.columns = ["year", "category", "game_name"]
        path = "Rule 2 (category from column)"

    elif not name_col:
        # ── RULE 3: melt ────────────────────────────────────
        value_cols = [c for c in t.columns if c != year_col]
        result = t.melt(
            id_vars=[year_col],
            value_vars=value_cols,
            var_name="category",
            value_name="game_name"
        ).rename(columns={year_col: "year"})
        path = f"Rule 3 (melt — cols: {value_cols})"

    else:
        # ── FALLBACK ────────────────────────────────────────
        print(f"  ⚠ [FALLBACK] Table {idx} ({award_name}): "
              f"fell through all rules — columns={t.columns.tolist()}")
        result = t[[year_col, name_col]].copy()
        result.columns = ["year", "game_name"]
        result["category"] = "General"
        path = "Fallback (General)"

    # ── CLEAN & FINALIZE ────────────────────────────────────
    result["year"]      = clean_citations(result["year"])
    result["year"]      = pd.to_numeric(result["year"], errors="coerce")
    result["game_name"] = clean_citations(result["game_name"])
    result["category"]  = clean_citations(result["category"])
    result["award"]     = award_name

    result = result.dropna(subset=["year", "game_name"])
    result = result[result["game_name"] != ""]
    result["year"] = result["year"].astype(int)

    # Reorder columns consistently
    result = result[["year", "award", "category", "game_name"]]

    award_frames.append(result)
    print(f"  ✓ Table {idx:2d} [{path}]: {award_name} / {category} — {len(result)} rows")

# Combine all award tables
awards_df = pd.concat(award_frames, ignore_index=True)
awards_df["game_name"] = awards_df["game_name"].str.strip()

/tmp/ipykernel_160/1255362836.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  all_tables = pd.read_html(r.text)
  ✓ Table  0 [Rule 1 (hardcoded category)]: American Tabletop Awards / Early Gamers — 7 rows
  ✓ Table  1 [Rule 1 (hardcoded category)]: American Tabletop Awards / Casual Games — 7 rows
  ✓ Table  2 [Rule 1 (hardcoded category)]: American Tabletop Awards / Strategy Games — 7 rows
  ✓ Table  3 [Rule 1 (hardcoded category)]: American Tabletop Awards / Complex Games — 7 rows
  ✓ Table  4 [Rule 2 (category from column)]: As d'Or / Golden Ace / General — 12 rows
  ⚠ [FALLBACK] Table 5 (Board Game Quest Awards): fell through all rules — columns=['year', 'winner', 'designer', 'publisher']
  ✓ Table  5 [Fallback (General)]: Board Game Quest Awards / General — 11 rows
  ⚠ [FALLBACK] Table 6 (Deutscher Spiele Preis): fell through all rules — columns=['year', 

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b3536b60-e481-4f67-b46c-1c4de87b554b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>